## Embeddings

In ons vorige voorbeeld werkten we met hoog-dimensionale bag-of-words vectoren van lengte `vocab_size`, en we converteerden expliciet van laag-dimensionale positionele representatievectoren naar een sparsere one-hot representatie. Deze one-hot representatie is niet geheugenefficiënt, bovendien wordt elk woord onafhankelijk van de anderen behandeld, d.w.z. one-hot gecodeerde vectoren drukken geen semantische gelijkenis tussen woorden uit.

In deze eenheid zullen we doorgaan met het verkennen van de **News AG** dataset. Om te beginnen, laten we de data laden en enkele definities ophalen uit het vorige notitieboek.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Wat is embedding?

Het idee van **embedding** is om woorden te representeren met lagere-dimensionale dichte vectoren, die op een of andere manier de semantische betekenis van een woord weerspiegelen. We zullen later bespreken hoe we betekenisvolle woord-embeddings kunnen bouwen, maar laten we voor nu embeddings zien als een manier om de dimensionaliteit van een woordvector te verlagen.

Dus, een embeddinglaag zou een woord als invoer nemen en een uitvoervector van gespecificeerde `embedding_size` produceren. In een zekere zin is het heel vergelijkbaar met een `Linear`-laag, maar in plaats van een one-hot gecodeerde vector te nemen, kan het een woordnummer als invoer nemen.

Door de embeddinglaag als eerste laag in ons netwerk te gebruiken, kunnen we overschakelen van een bag-of-words naar een **embedding bag**-model, waarbij we eerst elk woord in onze tekst omzetten in de bijbehorende embedding, en vervolgens een aggregatiefunctie over al die embeddings berekenen, zoals `sum`, `average` of `max`.

![Afbeelding die een embedding-classifier toont voor vijf sequentiewoorden.](../../../../../translated_images/nl/embedding-classifier-example.b77f021a7ee67eee.webp)

Ons classifier-neuraal netwerk begint met een embeddinglaag, daarna een aggregatielaag, en een lineaire classifier erbovenop:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Omgaan met variabele sequentiegrootte

Als gevolg van deze architectuur moeten minibatches voor ons netwerk op een bepaalde manier worden gemaakt. In de vorige eenheid, bij gebruik van bag-of-words, hadden alle BoW-tensors in een minibatch een gelijke grootte `vocab_size`, ongeacht de werkelijke lengte van onze tekstsequentie. Zodra we overstappen naar woord-embeddings, zouden we uitkomen op een variabel aantal woorden in elk tekstvoorbeeld, en bij het combineren van die voorbeelden in minibatches zouden we padding moeten toepassen.

Dit kan gedaan worden met dezelfde techniek van het aanleveren van een `collate_fn` functie aan de datasource:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Embedding-classifier trainen

Nu we een juiste datalader hebben gedefinieerd, kunnen we het model trainen met behulp van de trainingsfunctie die we in de vorige eenheid hebben gedefinieerd:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Opmerking**: We trainen hier slechts voor 25k records (minder dan één volledige epoch) vanwege tijdsredenen, maar je kunt de training voortzetten, een functie schrijven om voor meerdere epochs te trainen, en experimenteren met de parameter voor de leersnelheid om een hogere nauwkeurigheid te behalen. Je zou een nauwkeurigheid van ongeveer 90% moeten kunnen bereiken.


### EmbeddingBag Laag en Representatie van Sequenties van Variabele Lengte

In de vorige architectuur moesten we alle sequenties opvullen tot dezelfde lengte om ze in een minibatch te plaatsen. Dit is niet de meest efficiënte manier om sequenties van variabele lengte te representeren - een andere benadering zou zijn om een **offset** vector te gebruiken, die de offsets van alle sequenties opslaat in één grote vector.

![Afbeelding die een offset sequentieweergave toont](../../../../../translated_images/nl/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Opmerking**: Op de bovenstaande afbeelding tonen we een reeks tekens, maar in ons voorbeeld werken we met reeksen woorden. Het algemene principe van het representeren van reeksen met een offsetvector blijft echter hetzelfde.

Om met offsetrepresentatie te werken, gebruiken we de [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) laag. Deze lijkt op `Embedding`, maar neemt inhoudsvector en offsetvector als invoer, en bevat ook een averaging laag, die `mean`, `sum` of `max` kan zijn.

Hier is een aangepaste netwerk dat `EmbeddingBag` gebruikt:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Om de dataset voor te bereiden op training, moeten we een conversiefunctie leveren die de offsetvector voorbereidt:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Merk op dat, in tegenstelling tot alle voorgaande voorbeelden, ons netwerk nu twee parameters accepteert: datavector en offsetvector, die verschillende groottes hebben. Evenzo levert onze datalaadfunctie ons ook 3 waarden in plaats van 2: zowel tekst- als offsetvectoren worden als kenmerken geleverd. Daarom moeten we onze trainfunctie enigszins aanpassen om hiermee om te gaan:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Semantische Embeddings: Word2Vec

In ons vorige voorbeeld heeft de embeddinglaag van het model geleerd om woorden naar vectorrepresentaties te mappen, maar deze representatie had niet veel semantische betekenis. Het zou mooi zijn om zo'n vectorrepresentatie te leren, waarbij soortgelijke woorden of synoniemen overeenkomen met vectoren die dicht bij elkaar liggen qua vectordistantie (bijv. euclidische afstand).

Om dat te doen, moeten we ons embeddingmodel op een specifieke manier voortrainen op een grote verzameling tekst. Een van de eerste manieren om semantische embeddings te trainen heet [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Het is gebaseerd op twee hoofarchitecturen die worden gebruikt om een gedistribueerde representatie van woorden te produceren:

 - **Continue bag-of-words** (CBoW) — in deze architectuur trainen we het model om een woord te voorspellen vanuit de omliggende context. Gegeven de ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ is het doel van het model om $W_0$ te voorspellen vanuit $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** is het tegenovergestelde van CBoW. Het model gebruikt de omliggende venstercontextwoorden om het huidige woord te voorspellen.

CBoW is sneller, terwijl skip-gram langzamer is, maar beter presteert bij het representeren van weinig frequente woorden.

![Afbeelding die zowel de CBoW- als Skip-Gram-algoritmes toont om woorden naar vectoren om te zetten.](../../../../../translated_images/nl/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Om te experimenteren met word2vec-embedding voorgetraind op de Google News-dataset, kunnen we de **gensim**-bibliotheek gebruiken. Hieronder vinden we de woorden die het meest lijken op 'neural'.

> **Opmerking:** Bij het voor het eerst aanmaken van woordvectoren kan het downloaden enige tijd duren!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


We kunnen ook vector-embeddings berekenen van het woord, die gebruikt kunnen worden bij het trainen van een classificatiemodel (we tonen alleen de eerste 20 componenten van de vector voor duidelijkheid):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Het mooie van semantische embeddings is dat je vectorcodering kunt manipuleren om de semantiek te veranderen. We kunnen bijvoorbeeld vragen om een woord te vinden waarvan de vectorrepresentatie zo dicht mogelijk bij de woorden *king* en *woman* ligt, en zo ver mogelijk van het woord *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Zowel CBoW als Skip-Grams zijn “voorspellende” embeddings, omdat ze alleen lokale contexten in aanmerking nemen. Word2Vec maakt geen gebruik van globale context.

**FastText** bouwt voort op Word2Vec door vectorrepresentaties te leren voor elk woord en de karakter n-grammen die in elk woord voorkomen. De waarden van de representaties worden vervolgens gemiddeld tot één vector bij elke trainingsstap. Hoewel dit veel extra berekeningen toevoegt aan het voortrainen, maakt het woordembeddings mogelijk om subwoordinformatie te coderen.

Een andere methode, **GloVe**, maakt gebruik van het idee van een co-occurrencematrix en gebruikt neurale methoden om de co-occurrencematrix te ontleden in meer expressieve en niet-lineaire woordvectoren.

Je kunt spelen met het voorbeeld door de embeddings te veranderen naar FastText en GloVe, aangezien gensim verschillende woorden-embeddingmodellen ondersteunt.


## Het gebruik van voorgetrainde embeddings in PyTorch

We kunnen het bovenstaande voorbeeld aanpassen om de matrix in onze embeddinglaag vooraf te vullen met semantische embeddings, zoals Word2Vec. We moeten rekening houden met het feit dat de vocabularia van de voorgetrainde embedding en onze tekstcorpus waarschijnlijk niet overeenkomen, dus initialiseren we de gewichten voor de ontbrekende woorden met willekeurige waarden:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Laten we nu ons model trainen. Merk op dat de tijd die het kost om het model te trainen aanzienlijk groter is dan in het vorige voorbeeld, vanwege de grotere embedding-laaggrootte, en daardoor veel hoger aantal parameters. Ook kan het daardoor nodig zijn ons model op meer voorbeelden te trainen als we overfitting willen vermijden.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

In ons geval zien we geen grote toename in nauwkeurigheid, wat waarschijnlijk komt door heel verschillende vocabularia. 
Om het probleem van verschillende vocabularia te overwinnen, kunnen we een van de volgende oplossingen gebruiken:
* Het word2vec-model opnieuw trainen op onze vocabularium
* Onze dataset laden met het vocabularium van het voorgetrainde word2vec-model. Het vocabularium dat gebruikt wordt om de dataset te laden, kan worden gespecificeerd tijdens het laden.

De laatste aanpak lijkt makkelijker, vooral omdat het PyTorch `torchtext` framework ingebouwde ondersteuning voor embeddings bevat. We kunnen bijvoorbeeld een GloVe-gebaseerd vocabularium op de volgende manier instantieren:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Om de classifier te trainen met die embedding, moeten we eerst onze dataset coderen met behulp van de GloVe-woordenschat:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Zoals we hierboven hebben gezien, worden alle vectorinbeddingen opgeslagen in een `vocab.vectors` matrix. Dit maakt het heel eenvoudig om die gewichten in de gewichten van de inbeddingslaag te laden door eenvoudig te kopiëren:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Laten we nu ons model trainen en kijken of we betere resultaten krijgen:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Een van de redenen dat we geen significante stijging in nauwkeurigheid zien, is het feit dat sommige woorden uit onze dataset ontbreken in het voorgetrainde GloVe-woordenboek, en dus feitelijk worden genegeerd. Om dit probleem te overwinnen kunnen we onze eigen embeddings trainen op onze dataset.


## Contextuele Embeddings

Een belangrijke beperking van traditionele vooraf getrainde embedding representaties zoals Word2Vec is het probleem van woordbetekenisdisambiguering. Hoewel vooraf getrainde embeddings een deel van de betekenis van woorden in context kunnen vastleggen, wordt elke mogelijke betekenis van een woord in dezelfde embedding gecodeerd. Dit kan problemen veroorzaken in downstream modellen, omdat veel woorden zoals het woord 'play' verschillende betekenissen hebben afhankelijk van de context waarin ze worden gebruikt.

Bijvoorbeeld heeft het woord 'play' in deze twee verschillende zinnen een heel verschillende betekenis:
- Ik ging naar een **play** in het theater.
- John wil **playen** met zijn vrienden.

De bovenstaande vooraf getrainde embeddings vertegenwoordigen beide betekenissen van het woord 'play' in dezelfde embedding. Om deze beperking te overwinnen, moeten we embeddings bouwen op basis van het **taalmodel**, dat getraind is op een grote tekstcorpus en *weet* hoe woorden in verschillende contexten gecombineerd kunnen worden. Het bespreken van contextuele embeddings valt buiten de reikwijdte van deze tutorial, maar we komen er later op terug wanneer we het hebben over taalmodellen in de volgende eenheid.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
